# 1. See the data before training anything

MorphoFeatures starts *after* a 3D EM volume has been segmented. This tutorial slows
that hand-off down: we will inspect every input role, view the arrays in `(z, y, x)`,
check label alignment, and only then make a train/validation split.

**Questions answered here**

1. What do raw, cell, and nucleus arrays contain?
2. How do voxel coordinates become physical distances?
3. Why is `label_id` a join key rather than a row number?
4. Which checks should fail before a long cluster job is submitted?

The data are synthetic, so this notebook is safe on a laptop. Notebook 04 repeats the
same inspection on a bounded real Platynereis ROI.

## Step 0 — Reproducible setup

Run Jupyter from the repository root. Package helpers resolve the repository and output
root without relying on the notebook's current directory. The fixture is generated
beneath the configured output root; source data are never overwritten.

In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from morphofeatures.config import load_config, repository_root
from morphofeatures.data.crops import describe_labels
from morphofeatures.data.io import load_embeddings
from morphofeatures.data.synthetic import save_synthetic_dataset

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = repository_root()
CONFIG = load_config(REPO_ROOT / "configs" / "default.yaml")
OUTPUT_DIR = CONFIG.paths.output_root / "notebooks" / "01_data_contracts"
FIXTURE_DIR = save_synthetic_dataset(OUTPUT_DIR / "synthetic", seed=SEED)
print("Repository:", REPO_ROOT)
print("Generated fixture:", FIXTURE_DIR)

Repository: /g/kreshuk/frazer/miniforge3/envs/MorphoFeats_dev/lib/python3.11/site-packages
Generated fixture: /g/kreshuk/frazer/miniforge3/envs/MorphoFeats_dev/lib/python3.11/site-packages/outputs/notebooks/01_data_contracts/synthetic


## Step 1 — Identify the five input roles

| Input | What a voxel/row means | Essential rule |
|---|---|---|
| raw EM | image intensity | numeric and finite |
| cell segmentation | cell instance ID | integer; `0` is background |
| nucleus segmentation | nucleus instance ID | integer; `0` is background |
| cell–nucleus mapping | one biological association | join explicit IDs |
| metadata | annotation for one label | must contain `label_id` |

A real project may store the three volumes at different pyramid levels. Equal array
indices then do **not** imply equal physical positions: dataset key and resolution must
travel together.

In [ ]:
raw = np.load(FIXTURE_DIR / "raw.npy")
cells = np.load(FIXTURE_DIR / "cells.npy")
nuclei = np.load(FIXTURE_DIR / "nuclei.npy")
mapping = pd.read_csv(FIXTURE_DIR / "cell_to_nucleus.tsv", sep="	")
metadata = pd.read_csv(FIXTURE_DIR / "metadata.tsv", sep="	")

summary = pd.DataFrame(
    [
        ("raw", raw.shape, str(raw.dtype), float(raw.min()), float(raw.max()), len(np.unique(raw))),
        ("cells", cells.shape, str(cells.dtype), int(cells.min()), int(cells.max()), len(np.unique(cells))),
        ("nuclei", nuclei.shape, str(nuclei.dtype), int(nuclei.min()), int(nuclei.max()), len(np.unique(nuclei))),
    ],
    columns=["array", "shape_zyx", "dtype", "minimum", "maximum", "unique_values"],
)
display(summary)
display(mapping)
display(metadata)

## Step 2 — Look at aligned slices

The first array axis is **z** (section/depth), followed by **y** (row) and **x**
(column). The panels below use the same z index. Cell outlines should coincide with
image structures and nuclei should fall inside their mapped cells. A displaced overlay
is an alignment error, not something a model should be expected to repair.

In [ ]:
z = raw.shape[0] // 3
cell_boundaries = cells[z] != np.roll(cells[z], 1, axis=0)
cell_boundaries |= cells[z] != np.roll(cells[z], 1, axis=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(raw[z], cmap="gray")
axes[0].set_title(f"raw intensity, z={z}")
axes[1].imshow(cells[z], cmap="tab20", interpolation="nearest")
axes[1].set_title("cell label IDs")
axes[2].imshow(nuclei[z], cmap="tab20", interpolation="nearest")
axes[2].set_title("nucleus label IDs")
axes[3].imshow(raw[z], cmap="gray")
axes[3].contour(cell_boundaries, levels=[0.5], colors="cyan", linewidths=0.8)
axes[3].contour(nuclei[z] > 0, levels=[0.5], colors="magenta", linewidths=0.8)
axes[3].set_title("raw + cell/nucleus outlines")
for axis in axes:
    axis.set_axis_off()
plt.show()

### What this view can and cannot establish

It can reveal gross axis swaps, offsets, background mistakes, and obviously misplaced
nuclei. One slice cannot establish 3D segmentation quality. Inspect orthogonal planes,
small/large objects, disconnected fragments, and suspected merges in a volume viewer
before training on real data.

In [ ]:
label_table = describe_labels(cells)
resolution_zyx_um = np.array([0.10, 0.05, 0.05])  # illustrative fixture resolution
for axis, resolution in zip("zyx", resolution_zyx_um):
    label_table[f"extent_{axis}_um"] = (
        label_table[f"bb_max_{axis}"] - label_table[f"bb_min_{axis}"]
    ) * resolution

display(label_table[[
    "label_id", "voxel_count_roi", "touches_roi_border",
    "bb_min_z", "bb_min_y", "bb_min_x", "bb_max_z", "bb_max_y", "bb_max_x",
    "extent_z_um", "extent_y_um", "extent_x_um",
]])

## Step 3 — Inspect one biological unit in 3D

The next view isolates cell 1. The masked raw crop is the conceptual input to a coarse
cell-texture model: intensity inside one segmentation object, zero outside. Shape models
would instead derive a surface/point cloud from the binary mask. Fine-texture models
would sample multiple smaller high-resolution patches and aggregate them back to this
same `label_id`.

In [ ]:
label_id = 1
mask = cells == label_id
masked_raw = raw * mask
centers = np.rint(np.argwhere(mask).mean(axis=0)).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
views = [
    (raw[centers[0]], mask[centers[0]], masked_raw[centers[0]], "z"),
    (raw[:, centers[1]], mask[:, centers[1]], masked_raw[:, centers[1]], "y"),
]
for row, (raw_view, mask_view, masked_view, plane) in enumerate(views):
    axes[row, 0].imshow(raw_view, cmap="gray")
    axes[row, 0].set_title(f"raw ({plane} plane)")
    axes[row, 1].imshow(mask_view, cmap="gray")
    axes[row, 1].set_title("binary cell mask")
    axes[row, 2].imshow(masked_view, cmap="gray", vmin=raw.min(), vmax=raw.max())
    axes[row, 2].set_title("masked intensity")
for axis in axes.ravel():
    axis.set_axis_off()
plt.show()

## Step 4 — Audit IDs before relying on row order

IDs are semantic keys. Row 0 in one table is not automatically row 0 in another table.
The checks below expose missing segmentations, missing mappings, duplicate IDs, and
nuclei mapped to the wrong ID space.

In [ ]:
cell_ids = set(np.unique(cells)) - {0}
nucleus_ids = set(np.unique(nuclei)) - {0}
mapped_cells = set(mapping["cell_id"].astype(int))
mapped_nuclei = set(mapping["nucleus_id"].astype(int))
metadata_ids = set(metadata["label_id"].astype(int))

alignment_qc = pd.Series({
    "segmented cells missing from mapping": sorted(cell_ids - mapped_cells),
    "mapping cells missing from segmentation": sorted(mapped_cells - cell_ids),
    "mapped nuclei missing from segmentation": sorted(mapped_nuclei - nucleus_ids),
    "segmented cells missing metadata": sorted(cell_ids - metadata_ids),
    "duplicate mapping cell IDs": int(mapping["cell_id"].duplicated().sum()),
})
display(alignment_qc.to_frame("result"))
assert all(len(value) == 0 for value in alignment_qc.iloc[:4])
assert alignment_qc.iloc[4] == 0

## Step 5 — Read the label-first embedding contract literally

A NumPy embedding has shape `(n_cells, 1 + n_features)`. Column zero contains the
segmentation `label_id`; every remaining column is a learned feature. `label_id` must be
finite, integer-valued, unique, and joined explicitly to metadata.

In [ ]:
embedding = load_embeddings(FIXTURE_DIR / "embeddings.npy")
matrix = embedding.as_array()
print("matrix shape:", matrix.shape)
print("column 0 label IDs:", matrix[:, 0].astype(int).tolist())
print("feature matrix shape:", embedding.features.shape)
print("all features finite:", bool(np.isfinite(embedding.features).all()))

joined = pd.DataFrame({"label_id": embedding.label_ids}).merge(
    metadata, on="label_id", how="left", validate="one_to_one"
)
display(joined)

fig, axis = plt.subplots(figsize=(8, 3))
image = axis.imshow(embedding.features, aspect="auto", cmap="coolwarm")
axis.set_yticks(range(len(embedding.label_ids)), embedding.label_ids)
axis.set_xlabel("feature dimension")
axis.set_ylabel("label_id")
axis.set_title("Synthetic feature matrix (visual QC, not biology)")
fig.colorbar(image, ax=axis, label="feature value")
plt.show()

## Step 6 — Split IDs, then select rows

Splitting label IDs first makes the assignment reproducible and prevents a later table
sort from changing membership. For real biological data, also consider animal/batch,
spatial leakage, class balance, and related cells; a random cell split is not always an
independent biological test.

In [ ]:
train_ids, validation_ids = train_test_split(
    embedding.label_ids, test_size=0.5, random_state=SEED, shuffle=True
)
train_rows = np.isin(embedding.label_ids, train_ids)
validation_rows = np.isin(embedding.label_ids, validation_ids)
print("train label IDs:", sorted(train_ids.tolist()))
print("validation label IDs:", sorted(validation_ids.tolist()))
assert not np.any(train_rows & validation_rows)
assert np.all(train_rows | validation_rows)

## Replacing the fixture with cluster data

Set `MORPHOFEATURES_DATA_ROOT` instead of pasting a private path into source code. Record
the container path, internal dataset key, `(z, y, x)` resolution, segmentation version,
mapping/table version, crop policy, normalization, and selection exclusions in the run
directory. Then repeat the visual and ID audits above.

Notebook 04 demonstrates this exact transition for the locally available Platynereis
N5 data. It also shows why a promising dataset name is not enough: its shape, encoding,
ID space, and alignment must be inspected before use.